In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient, models
from pathlib import Path
import uuid
import gc

In [5]:
from config.paths import VECTOR_STORE

encoder = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cuda"
)

client = QdrantClient(path=VECTOR_STORE) # Carrega vectorstore em disco

Loading weights: 100%|██████████| 310/310 [00:00<00:00, 12322.51it/s]
/tmp/ipykernel_101145/2340899963.py:8: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Collection <Recurso_metadados> contains 63190 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client = QdrantClient(path=VECTOR_STORE) # Carrega vectorstore em disco


In [7]:
file_path = "/home/migueldcarvalho/Projetos/Chatbot/GovData/qr-janeiro-2026/qr-janeiro-2026.pdf"
#file_path = "/home/migueldcarvalho/Downloads/UERJ_2026.pdf"
collection_name = Path(file_path).name

In [ ]:
def embed_texts(texts, encoder, batch_size=5):
    vectors = []
    print(f"Tamanho do texto: {len(texts)}")
    for i in range(0, len(texts), batch_size):
        print(i)
        batch = texts[i:i + batch_size]
        emb = encoder.encode(batch, convert_to_numpy=True)
        vectors.extend(emb)
        del batch, emb
        gc.collect()
    return vectors

def external_doc(config):
    client = config["client"]
    encoder = config["encoder"]
    file_path = config['file_path']

    loader = PyPDFLoader(file_path)
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = text_splitter.split_documents(docs)
    texts = [chunk.page_content for chunk in chunks]

    vectors = embed_texts(texts, encoder, batch_size=5)

    if not client.collection_exists(collection_name=collection_name):
        client.create_collection(
            collection_name=collection_name, # nome da coleção
            vectors_config=models.VectorParams(
            size=encoder.get_sentence_embedding_dimension(),  # Vector size is defined by used model
            distance=models.Distance.COSINE, # Metrica de similaridade
        ),
)

    points = []
    for i, chunk in enumerate(chunks):
        points.append(
            models.PointStruct(
                id=str(uuid.uuid4()),
                vector=vectors[i].tolist(),
                payload={
                    "page_content": chunk.page_content,
                    "metadata": chunk.metadata,
                    "chunk_index": i
                }
            )
        )

    client.upsert(
        collection_name=collection_name,
        points=points
    )

In [ ]:
#query = "Qual é o dia de volta as aulas no primeiro periodo?"
#query = "Quero informações sobre INSPEÇÃO DE EQUIPAMENTOS E INSTALAÇÕES"

state = {
    "client": client,
    "encoder": encoder,
    "file_path": file_path
}

if not client.collection_exists(collection_name=collection_name):
        external_doc(state)


Tamanho do texto: 27
0
5
10
15
20
25


In [ ]:
queries = ["Quero informações sobre os requisitos necessário para enfermagem do trabalho",
           "Dados sobre manutenção calderaria",
           "Quero informações sobre INSPEÇÃO DE EQUIPAMENTOS E INSTALAÇÕES",
           "Sobre o que é o edital?",
           "Quais são as datas mais importantes?",
           "Qual é a banca deste concurso?",
           "Quero informações das vagas de PCDs"]

In [11]:
query = "Há quantos cargos de administrador"

hits = client.query_points(
    collection_name=collection_name,
    query=encoder.encode(query).tolist(),
    limit=5
).points

for hit in hits:
    print(hit.payload['page_content'])
    print()

__ SIAPE,GERENCIAL,GRVAGACARG,GRCOLOTREA ( CONSULTA LOTACAO REAL )____________
  DATA: 13JAN2026    HORA: 14:01:30    USUARIO: ALISSON VANDRE          PRODUCAO
  ORGAO: 26402 - IFAL                                      MES TABELA :  JAN2026
                                                                                
  ORGAO: 26402 - INSTITUTO FEDERAL DE ALAGOAS                                   
  GRUPO: 701 - PLANO DE CARREIRA DOS CARGOS TAE-IFE                             
  CARGO: TODOS                                                                  
  ------------------------------------------------------------------------------
  CARGO                                        ESCOL   VAGOS   OCUPADOS    TOTAL
  ------------------------------------------------------------------------------
  001 ADMINISTRADOR                             NS                   23       23
  004 ARQUITETO E URBANISTA                     NS                    4        4

001 ADMINISTRADOR            

In [6]:
from chatbot.tools.pdf_reader import external_doc
from pathlib import Path
import os

file_path = "/home/migueldcarvalho/Projetos/Chatbot/GovData/qr-janeiro-2026/qr-janeiro-2026.pdf"
collection_name = Path(os.path.basename(file_path)).stem

config = {
    "client": client,
    "encoder": encoder,
    "file_path": file_path
}

external_doc(config)


Tamanho do texto: 27
0
5
10
15
20
25


In [7]:
query = "Há quantos cargos de administrador"

hits = client.query_points(
    collection_name=collection_name,
    query=encoder.encode(query).tolist(),
    limit=5
).points

for hit in hits:
    print(hit.payload['page_content'])
    print()

__ SIAPE,GERENCIAL,GRVAGACARG,GRCOLOTREA ( CONSULTA LOTACAO REAL )____________
  DATA: 13JAN2026    HORA: 14:01:30    USUARIO: ALISSON VANDRE          PRODUCAO
  ORGAO: 26402 - IFAL                                      MES TABELA :  JAN2026
                                                                                
  ORGAO: 26402 - INSTITUTO FEDERAL DE ALAGOAS                                   
  GRUPO: 701 - PLANO DE CARREIRA DOS CARGOS TAE-IFE                             
  CARGO: TODOS                                                                  
  ------------------------------------------------------------------------------
  CARGO                                        ESCOL   VAGOS   OCUPADOS    TOTAL
  ------------------------------------------------------------------------------
  001 ADMINISTRADOR                             NS                   23       23
  004 ARQUITETO E URBANISTA                     NS                    4        4

001 ADMINISTRADOR            